<a href="https://colab.research.google.com/github/mahoneyjustinnj/myGarminHealth_JM/blob/main/analyzingGarminHeartDataColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# from google.colab import ai
# response = ai.generate_text("What is the capital of France?")

In [1]:
seconds_in_a_day = 24 * 60 * 60
seconds_in_a_day

[1] 86400

In [ ]:
#$$
suppressMessages(install.packages("tidyverse"))
suppressMessages(install.packages("lubridate"))
suppressMessages(install.packages("FSA"))
suppressMessages(install.packages("RColorBrewer"))
suppressMessages(install.packages("dplyr"))
suppressMessages(install.packages("plotly"))
suppressMessages(install.packages("ggplot2"))
suppressMessages(install.packages("ggpubr"))
suppressMessages(install.packages("skimr"))
suppressMessages(install.packages("tibble"))
suppressMessages(install.packages("htmltools"))
suppressMessages(install.packages("tibbletime"))
suppressMessages(install.packages("zoo"))
suppressMessages(install.packages("tidyr"))
suppressMessages(install.packages("anomalize"))
suppressMessages(install.packages("knitr"))
suppressMessages(install.packages("gt"))
suppressMessages(install.packages("stringr"))
suppressMessages(install.packages("grid"))
#$$
library(tidyverse)       # Data manipulation and visualization (includes dplyr, ggplot2, etc.)
library(lubridate)       # Date and time manipulation functions
library(FSA)             # Fisheries Stock Assessment - includes statistical tests like Dunn's test
library(RColorBrewer)    # Color palettes for plots
library(dplyr)           # Data frame manipulation (part of tidyverse, loaded explicitly for clarity)
library(plotly)          # Interactive plotting library
library(ggplot2)         # Static plotting (part of tidyverse, loaded explicitly)
library(ggpubr)          # Publication-ready ggplot2 figures
library(skimr)           # Quick data summary function
library(tibble)          # Modern data frame format
library(htmltools)       # HTML utilities for Shiny/markdown
library(tibbletime)      # Time-aware tibbles for time series
library(zoo)             # Time series and irregular data handling (includes na.approx)
library(tidyr)           # Data reshaping (pivot_wider, unnest, etc.)
library(anomalize)       # Time series anomaly detection
library(knitr)           # R Markdown processing
library(gt)              # Great Tables - nice table formatting
library(stringr)         # String manipulation functions
library(grid)            # Low-level graphics utilities
#$$
knitr::opts_chunk$set(
  warning = FALSE,    # Suppress warning messages during knitting
  message = FALSE     # Suppress informational messages during knitting
)



In [ ]:
#$$
# cat("\014")  # Clear console (commented out)
gc()                                                    # Garbage collection - free up memory

rm(list = ls())                                         # Remove all objects from environment

setwd("/content")

In [ ]:
#$$
df_raw <- read_csv("heart22.csv")
df <- df_raw
#$$
# Display first n rows of dataframe with gt styling and header
show_tb <- function(df, n = 6) {
  library(gt)

  df %>%
    head(n) %>%                          # Get first n rows
    gt() %>%                             # Convert to gt table object
    tab_header(
      title = paste("Showing First", n, "Rows"),      # Add title with dynamic row count
      subtitle = "Styled with gt"                      # Add subtitle
    )
}
#$$
# Track dataframes and their columns in all_names data structure
fx <- function(...  ) {
  arg_exprs <- substitute(list(...))[-1]  # Capture unevaluated arguments as expressions
  i <- 1                                   # Initialize loop counter

  while (i <= length(arg_exprs)) {
    expr <- arg_exprs[[i]]                 # Get current expression
    is_comment <- FALSE                    # Flag for comment detection (unused)

    # Skip comment-only arguments
    if (is.character(expr)) {
      i <- i + 1
      next                                 # Move to next iteration if argument is character string
    }

    df_name <- deparse(expr)               # Convert expression to character string (dataframe name)
    comment <- ""                          # Initialize comment variable

    # Check if next argument is a comment string
    if (i < length(arg_exprs) && is.character(arg_exprs[[i + 1]])) {
      comment <- eval(arg_exprs[[i + 1]], envir = .GlobalEnv)  # Evaluate next arg as comment
      i <- i + 1                           # Skip the comment in next iteration
    }

    # Evaluate dataframe existence and get column names
    df_exists <- exists(df_name, envir = .GlobalEnv)  # Check if dataframe exists in global environment
    if (df_exists) {
      df <- get(df_name, envir = .GlobalEnv)         # Retrieve the dataframe object
      if (!inherits(df, "data.frame")) {
        message(df_name, " exists but is not a dataframe.   Skipping.")
        i <- i + 1
        next                               # Skip if not a dataframe
      }
      col_list <- paste(names(df), collapse = ", ")  # Get all column names as comma-separated string
    } else {
      col_list <- ""                       # Empty string if dataframe doesn't exist
      comment <- paste0("object '", df_name, "' not found\n", comment)  # Add error message to comment
      message(df_name, " is not present in environment but added to all_names.")
    }

    # Add or update entry in tracking dataframe
    idx <- which(all_names$dataframe == df_name)     # Find if dataframe already tracked
    if (length(idx) == 1) {
      if (comment != "") {
        all_names$comment[idx] <<- paste(all_names$comment[idx], comment, sep = "\n")  # Append comment
      }
      if (col_list != "") {
        all_names$columns[idx] <<- col_list          # Update columns if not empty
      }
    } else {
      all_names <<- rbind(
        all_names,
        data.frame(dataframe = df_name, columns = col_list, comment = comment, stringsAsFactors = FALSE)
      )                                    # Add new row to all_names tracking table
    }

    i <- i + 1                             # Increment counter
  }

  # Reorder:  move rows with empty columns to bottom
  empty_rows <- which(all_names$columns == "")       # Find rows with no columns (missing dataframes)
  filled_rows <- which(all_names$columns != "")      # Find rows with columns (existing dataframes)
  all_names <<- all_names[c(filled_rows, empty_rows), ]  # Reorder:  existing first, missing last
}

#$$
# Compare column names between dataframes
comp_cols <- function(df_raw, ..., reverse = FALSE) {
  # Capture exclusions as character strings
  exclude <- as.character(substitute(list(... )))[-1]

  # Get column names from df_raw
  df_names <- names(df_raw)

  # Filter all_names based on exclude list
  if (length(exclude) > 0) {
    all_names_filtered <- all_names[!(all_names[[1]] %in% exclude), ]
  } else {
    all_names_filtered <- all_names
  }

  # Parse and clean all comma-separated column names
  ref_names <- unlist(strsplit(paste(all_names_filtered[[2]], collapse = ","), ","))
  ref_names_clean <- trimws(ref_names)

  # Return either missing or extra columns based on reverse flag
  if (reverse) {
    return(ref_names_clean[!(ref_names_clean %in% df_names)])
  } else {
    return(df_names[!(df_names %in% ref_names_clean)])
  }
}

#$$
# Initialize empty tracking dataframe
all_names <- data.frame(
  dataframe = character(0),
  columns = character(0),
  comment = character(0),
  stringsAsFactors = FALSE
)

#$$
# Add or append comment to tracked dataframe
add_comment <- function(df, comment) {
  # Convert dataframe argument to string to get its name
  df_name <- deparse(substitute(df))

  # Search all_names for row matching the dataframe name
  idx <- which(all_names$dataframe == df_name)

  # If dataframe found in all_names (should be exactly one match)
  if (length(idx) == 1) {
    # Check if comment field is currently empty
    if (all_names$comment[idx] == "") {
      # If empty, assign comment directly to the dataframe row
      all_names$comment[idx] <<- comment
    } else {
      # If comment already exists, append new comment with newline separator
      all_names$comment[idx] <<- paste(all_names$comment[idx], comment, sep = "\n")
    }
  }
}
#$$
# Clear all tracked dataframes in all_names
reset_fx <- function() {
  # Reassign all_names to empty dataframe with same structure
  all_names <<- data.frame(
    dataframe = character(0),  # Empty character vector for dataframe names
    columns = character(0),    # Empty character vector for column lists
    comment = character(0),    # Empty character vector for comments
    stringsAsFactors = FALSE   # Prevent automatic conversion of strings to factors
  )
}

#$$
# Display comments for a specific dataframe
df_comms <- function(df) {
  # Convert dataframe argument to string to get its name
  df_name <- deparse(substitute(df))

  # Search all_names for row matching the dataframe name
  idx <- which(all_names$dataframe == df_name)

  # If dataframe not found in all_names or comment field is empty
  if (length(idx) == 0 || all_names$comment[idx] == "") {
    # Print message indicating no comments exist for this dataframe
    message("There are no comments currently for this dataframe.")
  } else {
    # If comments exist, print them to console
    cat(all_names$comment[idx])
  }
}

#$$
# List all current dataframe objects in environment
curr_df <- function(env = .GlobalEnv) {
  # Get list of all object names in the specified environment
  objs <- ls(envir = env)

  # Filter objects to keep only those that are dataframes or dataframe-like objects
  df_objs <- Filter(function(x) {
    # Retrieve object from environment
    obj <- get(x, envir = env)

    # Check if object inherits from data.frame, tbl_df, or data.table classes
    inherits(obj, c("data.frame", "tbl_df", "data.table"))
  }, objs)

  # If no dataframes found in environment
  if (length(df_objs) == 0) {
    # Print message indicating no data-holding objects were found
    message("No data-holding objects found.")
  } else {
    # If dataframes found, print their names to console
    print(df_objs)
  }
}

#$$
# Find untracked dataframes in environment
misd_df <- function() {
  # Get current data-holding objects from environment
  current_dfs <- curr_df()

  # If no dataframes exist, return invisibly without printing anything
  if (is.null(current_dfs)) return(invisible(NULL))

  # Get list of already tracked dataframes from all_names
  tracked_dfs <- all_names$dataframe

  # Find dataframes that exist but are not in all_names (untracked)
  missing <- setdiff(current_dfs, tracked_dfs)

  # If all dataframes are already tracked
  if (length(missing) == 0) {
    # Print message indicating all dataframes are tracked
    message("All current dataframes are already tracked in all_names.")
  } else {
    # If untracked dataframes found, print header message
    cat("Untracked dataframes:\n")

    # Print list of untracked dataframes
    print(missing)
  }
}

#$$
# Extract calendar date and heart rate values columns
dtwp <- df %>%
  select(calendarDate, heartRateValues)  # Select only these two columns from df

# Track dtwp dataframe with description
fx(dtwp, "takes these 2 columns from df")

# Extract all columns except heart rate detail columns
de <- df %>%
  select(-heartRateValues, -heartRateValueDescriptors)  # Remove heart rate detail columns

# Track de dataframe with description
fx(de, "de takes everything from df except heartRateValues and heartRateValueDescriptors")

#$$
# Replace 'None' with NULL and parse JSON heart rate values
dtwp1 <- dtwp %>%
  mutate(heartRateValues = map(heartRateValues, ~ {
    # Replace Python None with JSON null for proper JSON parsing
    .x <- gsub("None", "null", .x)

    # Parse the JSON string into R data structure
    parsed_data <- jsonlite::fromJSON(.x)

    # Return parsed data
    return(parsed_data)
  }))

# Track dtwp1 dataframe with description
fx(dtwp1, "dtwp1 replaces 'None' with NULL and parses JSON from dtwp")

#$$
# Unnest heart rate values from list to individual rows
dtwp2 <- dtwp1 %>%
  unnest(cols = heartRateValues)  # Convert nested list column into multiple rows with individual values

# Track dtwp2 dataframe with description
fx(dtwp2, "dtwp2 unnests heartRateValues into milliseconds and heart rate")

#$$
# Convert to dataframe and rename columns for clarity
data <- data.frame(dtwp2$calendarDate, dtwp2$heartRateValues[, 1], dtwp2$heartRateValues[, 2])
# Extract calendar date from dtwp2 as first column
# Extract first column from heartRateValues (milliseconds/seconds timestamp)
# Extract second column from heartRateValues (heart rate value)

# Rename columns with more descriptive names
colnames(data) <- c("date", "seconds", "heartrate")
# "date" - calendar date of heart rate measurement
# "seconds" - Unix timestamp in milliseconds
# "heartrate" - heart rate value in beats per minute

# Track data dataframe with description
fx(data, "data renames columns and converts to dataframe with proper column names")

#$$
# Convert Unix timestamp (seconds) to POSIXct datetime
data$time <- as.POSIXct(data$seconds / 1000, origin = "1970-01-01", tz = "UTC")

# Convert UTC time to Eastern Time (handles daylight saving automatically)
data$time <- with_tz(data$time, tzone = "America/Toronto")

# Handle missing values with linear interpolation
data <- data %>% mutate(heartrate = na.approx(heartrate, na.rm = FALSE))

# Track data dataframe with timezone conversion description
fx(data, "data now includes Eastern Time Zone conversion and date extraction")

#$$
# Remove original UTC datetime column as it's no longer needed
data <- data %>%
  select(-datetime)  # Drop the UTC datetime column to reduce redundancy

# Rename Eastern Time datetime column to primary datetime column
data <- data %>%
  rename(datetime = datetime_et)  # Rename datetime_et to datetime for simplicity

# Track data dataframe with cleanup description
fx(data, "data cleaned up:  removed UTC datetime, renamed datetime_et to datetime")

#$$
# Display first few rows of processed data
show_tb(data, 10)
#$$
# Check the date range in your data
cat("Data date range:\n")
cat("Min date:", format(min(data$time, na.rm = TRUE), "%Y-%m-%d"), "\n")
cat("Max date:", format(max(data$time, na.rm = TRUE), "%Y-%m-%d"), "\n")
cat("Total rows:", nrow(data), "\n")
#$$
# Extract daily heart rate statistics from de dataframe
de1 <- de %>% select(calendarDate, maxHeartRate, minHeartRate, restingHeartRate, lastSevenDaysAvgRestingHeartRate)

# Track de1 dataframe with description
fx(de1, "de1 contains daily heart rate statistics (max, min, resting, 7-day avg)")
#$$
# Extract HRV summary and reading columns
dg <- de %>% select(-setdiff(names(de1), "calendarDate"))
fx(dg, "dg contains all columns from de except those in de1 (excluding calendarDate)")

dg1 <- dg %>% select(c("calendarDate", "summaryDate", "userProfilePK",
                       "startTimestampGMT", "endTimestampGMT",
                       "startTimestampLocal", "endTimestampLocal",
                       "userProfilePk", "hrvSummary", "hrvReadings"))
fx(dg1, "dg1 subsets specific HRV-related columns from dg")

dg2 <- dg1 %>% select(c("calendarDate", "hrvSummary"))
dg3 <- dg1 %>% select(c("calendarDate", "hrvReadings"))
#$$
# Parse HRV summary JSON data
colnames(dg2) <- c("CalendarDate", "hrvSummary")
dg2_proc <- dg2 %>%
  mutate(hrvSummary = map(hrvSummary, ~ {
    # Replace Python syntax with JSON syntax
    .x <- gsub("None", "null", .x)
    .x <- gsub("'", "\"", .x)
    jsonlite::fromJSON(.x)
  })) %>%
  unnest_longer(hrvSummary, values_to = "value", indices_to = "key")
fx(dg2_proc, "dg2_proc parses JSON from dg2 and converts to long format")

# Pivot to wide format for cleaner dataframe
dg2_flat <- dg2_proc %>%
  pivot_wider(names_from = key, values_from = value)
fx(dg2_flat, "dg2_flat flattens dg2_proc into separate columns per HRV metric")

#$$
# Parse HRV readings (individual measurements throughout day)
dg3_flat <- dg3 %>%
  mutate(hrvReadings = map(hrvReadings, ~ {
    .x <- gsub("None", "null", .x)
    .x <- gsub("'", "\"", .x)
    jsonlite::fromJSON(.x)
  })) %>%
  # Map over each reading list and add calendarDate to each row
  { purrr::map2_dfr(.$calendarDate, .$hrvReadings,
                    ~ dplyr::mutate(as.data.frame(.y), calendarDate = .x)) }
fx(dg3_flat, "dg3_flat extracts individual HRV readings from list-of-dicts column")

# Rename for consistency with other datasets
daty <- dg3_flat
fx(daty, "daty is identical to dg3_flat - used for HRV time series analysis")

#$$
# Convert HRV reading times from ISO format to POSIXct
daty$readingTimeGMT <- as.POSIXct(daty$readingTimeGMT, format = "%Y-%m-%dT%H:%M:%OS", tz = "UTC")

# Convert to Eastern Time
daty$readingTimeLocal <- with_tz(daty$readingTimeGMT, tzone = "America/Toronto")

#$$
# Define holidays for visual reference
holidays <- as.Date(c("2025-01-01", "2025-07-01", "2025-12-25"))

# Create summary of daily time ranges and markers
daily_lines <- daty %>%
  group_by(calendarDate) %>%
  summarise(
    x_start = min(readingTimeLocal),
    x_end = max(readingTimeLocal),
    .groups = "drop"
  ) %>%
  mutate(
    is_weekend = wday(calendarDate) %in% c(1, 7),
    is_holiday = calendarDate %in% holidays,
    label_color = ifelse(is_weekend | is_holiday, "#E74C3C", "#2980B9")
  )
fx(daily_lines, "daily_lines contains date markers for HRV plot (weekends/holidays highlighted)")
